# Resultados da ablação — sMCI vs pMCI

Pipeline de visualização para apresentação e artigo.

**Entrada:** `ablation_summary_*.csv` e `ablation_results_*.csv` em `csvs/abordagem_4_sMCI_pMCI_extremos/`

**Saída:** figuras PNG (300 dpi) e PDF em `csvs/abordagem_4_sMCI_pMCI_extremos/figures/`

**Protocolo:** nested CV 5×5, seleção MRMR por bloco (corr + var + MRMR), modelos SVM / RF / MLP.

# Filters

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

RESULTS_DIR = Path("csvs/longitudinal_4_groups/ablation_results")
RESULTS_PATH = RESULTS_DIR / "ablation_results_all_filters.csv"
SUMMARY_PATH = RESULTS_DIR / "ablation_summary_filters.csv"


MODEL_ORDER = ["svm", "rf", "mlp"]
MOD_ORDER = ["vol", "texture", "disp", "all"]
COMBAT_ORDER = ["sem ComBat", "ComBat"]
COLOR_MAP = {"svm": "#4477AA", "rf": "#EE6677", "mlp": "#228833"}


def _coerce_with_combat(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().isin(("true", "1", "yes"))


def _load_summary_if_needed() -> pd.DataFrame:
    if "summary" in globals() and globals().get("summary") is not None:
        return globals()["summary"]
    if "SUMMARY_PATH" not in globals():
        raise NameError("Rode a célula de config (imports + RESULTS_DIR).")
    summary_path = globals()["SUMMARY_PATH"]
    if not Path(summary_path).is_file():
        raise FileNotFoundError(f"summary ausente: {summary_path}")
    return pd.read_csv(summary_path)


summary = _load_summary_if_needed()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plot_df = summary.copy()
if "with_combat" not in plot_df.columns:
    raise KeyError("summary sem coluna with_combat")
plot_df["with_combat"] = _coerce_with_combat(plot_df["with_combat"])
plot_df["combat_label"] = plot_df["with_combat"].map({True: "ComBat", False: "sem ComBat"})
if plot_df["combat_label"].isna().any():
    raise ValueError("with_combat inválido — valores não mapeiam para ComBat/sem ComBat")

plot_df["model_key"] = pd.Categorical(
    plot_df["model_key"].astype(str), categories=MODEL_ORDER, ordered=True
)
plot_df["modality"] = pd.Categorical(
    plot_df["modality"].astype(str), categories=MOD_ORDER, ordered=True
)

tasks = sorted(plot_df["task"].astype(str).unique())
combat_labels = [c for c in COMBAT_ORDER if c in set(plot_df["combat_label"])]
if not tasks or not combat_labels:
    raise ValueError(f"Nada para plotar: tasks={tasks}, combat={combat_labels}")

# 1) Heatmap AUC: modalidade × modelo (painel por task × ComBat)
fig, axes = plt.subplots(
    len(tasks), len(combat_labels),
    figsize=(4 * len(combat_labels), 3 * len(tasks)),
    squeeze=False,
)
for i, task_id in enumerate(tasks):
    for j, combat in enumerate(combat_labels):
        ax = axes[i, j]
        sub = plot_df[(plot_df["task"].astype(str) == task_id) & (plot_df["combat_label"] == combat)]
        if sub.empty:
            ax.axis("off")
            continue
        pivot = sub.pivot_table(
            index="model_key", columns="modality", values="auc_mean", aggfunc="first", observed=False
        )
        pivot = pivot.reindex(index=MODEL_ORDER, columns=MOD_ORDER)
        sns.heatmap(
            pivot, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0.5, vmax=1.0,
            ax=ax, cbar=(i == 0 and j == len(combat_labels) - 1),
            cbar_kws={"label": "AUC"} if (i == 0 and j == len(combat_labels) - 1) else None,
        )
        ax.set_title(f"{task_id}\n{combat}", fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("modelo" if j == 0 else "")
fig.suptitle("AUC médio (5 folds)", fontsize=12)
plt.tight_layout()
out_heat = RESULTS_DIR / "ablation_summary_heatmap_auc.png"
# fig.savefig(out_heat, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura: {out_heat}")

# 2) AUC com barras + std (um gráfico por task)
for task_id in tasks:
    sub = plot_df[plot_df["task"].astype(str) == task_id]
    fig, axes = plt.subplots(1, len(combat_labels), figsize=(5 * len(combat_labels), 4), squeeze=False)
    for ax, combat in zip(axes.flat, combat_labels):
        part = sub[sub["combat_label"] == combat]
        labels, means, stds, colors = [], [], [], []
        for mod in MOD_ORDER:
            for model in MODEL_ORDER:
                row = part[(part["modality"].astype(str) == mod) & (part["model_key"].astype(str) == model)]
                if row.empty:
                    continue
                r = row.iloc[0]
                labels.append(f"{mod}\n{model}")
                means.append(float(r["auc_mean"]))
                stds.append(float(r["auc_std"]))
                colors.append(COLOR_MAP[model])
        if labels:
            ax.bar(labels, means, yerr=stds, capsize=3, color=colors, ecolor="gray")
        else:
            ax.text(0.5, 0.5, "sem dados", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(combat)
        ax.set_ylabel("AUC")
        ax.tick_params(axis="x", labelsize=7)
        ax.axhline(0.5, color="gray", ls="--", lw=0.8)
        ax.set_ylim(0.4, 1.0)
    fig.suptitle(f"Task {task_id} — AUC ± std", fontsize=11)
    plt.tight_layout()
    out = RESULTS_DIR / f"ablation_summary_bars_{task_id}.png"
    # fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figura: {out}")

# 3) Melhor config por task × modalidade
best = (
    plot_df.sort_values("auc_mean", ascending=False)
    .groupby(["task", "modality", "combat_label"], as_index=False, observed=True)
    .first()
)
fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * len(best))))
y_labels = (
    best["task"].astype(str) + " | " + best["modality"].astype(str) + " | " + best["combat_label"]
)
ax.barh(
    range(len(best)),
    best["auc_mean"],
    xerr=best["auc_std"],
    color=best["model_key"].astype(str).map(COLOR_MAP),
    capsize=3,
    ecolor="gray",
)
ax.set_yticks(range(len(best)))
ax.set_yticklabels(y_labels, fontsize=8)
ax.set_xlabel("AUC médio ± std")
ax.axvline(0.5, color="gray", ls="--", lw=0.8)
ax.set_title("Melhor modelo (maior AUC) por task × modalidade")
ax.legend(handles=[Patch(facecolor=c, label=k.upper()) for k, c in COLOR_MAP.items()])
plt.tight_layout()
out_best = RESULTS_DIR / "ablation_summary_best_auc.png"
# fig.savefig(out_best, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura: {out_best}")


In [ ]:
# Principais atributos por tarefa de classificação — complemento aos gráficos de AUC
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# alinhar com célula de gráficos AUC
RESULTS_DIR = Path("csvs/longitudinal_4_groups/ablation_results")
RESULTS_PATH = RESULTS_DIR / "ablation_results_all_filters.csv"

MOD_ORDER = ["vol", "texture", "disp", "all"]
COMBAT_ORDER = ["sem ComBat", "ComBat"]
TOP_N = 12          # atributos por painel
FEAT_RE = re.compile(r"^hippocampus_([LR])_(T[123])_(.+)$")


def _coerce_with_combat(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().isin(("true", "1", "yes"))


def _short_feature(name: str) -> str:
    m = FEAT_RE.match(name)
    if m:
        return f"{m.group(1)} {m.group(2)} | {m.group(3)}"
    return name


def _load_ablation() -> pd.DataFrame:
    if "df_ablation" in globals() and globals()["df_ablation"] is not None:
        df = globals()["df_ablation"].copy()
    else:
        if not RESULTS_PATH.is_file():
            raise FileNotFoundError(f"CSV ausente: {RESULTS_PATH}")
        df = pd.read_csv(RESULTS_PATH)
    if "selected_features" not in df.columns:
        raise KeyError("Coluna selected_features ausente — rode ablação com runner atual.")
    df["with_combat"] = _coerce_with_combat(df["with_combat"])
    df["combat_label"] = df["with_combat"].map({True: "ComBat", False: "sem ComBat"})
    return df


def feature_freq_table(df: pd.DataFrame, top_n: int = TOP_N) -> pd.DataFrame:
    """Frequência de cada atributo em selected_features (por linha do CSV)."""
    rows = []
    for _, r in df.iterrows():
        for feat in json.loads(r["selected_features"]):
            rows.append({
                "task": r["task"],
                "modality": r["modality"],
                "combat_label": r["combat_label"],
                "feature": feat,
                "feature_short": _short_feature(feat),
            })
    if not rows:
        return pd.DataFrame(columns=["task", "modality", "combat_label", "feature", "feature_short", "count", "freq"])

    long = pd.DataFrame(rows)
    n_runs = len(df)
    agg = (
        long.groupby(["task", "modality", "combat_label", "feature", "feature_short"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
    )
    agg["freq"] = agg["count"] / n_runs
    agg = agg.sort_values(["task", "modality", "combat_label", "freq"], ascending=[True, True, True, False])
    agg["rank"] = agg.groupby(["task", "modality", "combat_label"]).cumcount() + 1
    return agg[agg["rank"] <= top_n]


def plot_top_features_by_task_modality(df_ablation: pd.DataFrame) -> None:
    """
  Fig 1 — espelha barras de AUC: 1 figura por task, painéis ComBat,
  barras horizontais = top atributos por modalidade.
  """
    freq = feature_freq_table(df_ablation, top_n=TOP_N)
    tasks = sorted(df_ablation["task"].astype(str).unique())
    combat_labels = [c for c in COMBAT_ORDER if c in set(df_ablation["combat_label"])]

    for task_id in tasks:
        sub = freq[freq["task"].astype(str) == task_id]
        if sub.empty:
            continue

        fig, axes = plt.subplots(
            1, len(combat_labels),
            figsize=(6 * len(combat_labels), 5),
            squeeze=False,
        )
        for ax, combat in zip(axes.flat, combat_labels):
            part = sub[sub["combat_label"] == combat]
            if part.empty:
                ax.axis("off")
                continue

            y_pos = 0
            yticks, ylabels = [], []
            for mod in MOD_ORDER:
                block = part[part["modality"].astype(str) == mod].sort_values("freq", ascending=False)
                if block.empty:
                    continue
                for _, row in block.iterrows():
                    ax.barh(y_pos, row["freq"], height=0.8, alpha=0.85, label=mod if y_pos == 0 else None)
                    yticks.append(y_pos)
                    ylabels.append(f"{mod} | {row['feature_short']}")
                    y_pos += 1
                y_pos += 0.4  # espaço entre modalidades

            ax.set_yticks(yticks)
            ax.set_yticklabels(ylabels, fontsize=7)
            ax.set_xlim(0, 1.05)
            ax.set_xlabel("Frequência de seleção")
            ax.set_title(combat)
            ax.axvline(0.5, color="gray", ls="--", lw=0.6, alpha=0.5)

        fig.suptitle(f"Top atributos — task {task_id}", fontsize=11)
        plt.tight_layout()
        out = RESULTS_DIR / f"ablation_features_bars_{task_id}.png"
        plt.show()
        print(f"Figura: {out}")


def plot_feature_heatmap(df_ablation: pd.DataFrame) -> None:
    """
  Fig 2 — espelha heatmap de AUC: painéis task × ComBat,
  colunas = modalidade, linhas = atributo (nome curto), cor = frequência.
  """
    freq = feature_freq_table(df_ablation, top_n=TOP_N)
    tasks = sorted(df_ablation["task"].astype(str).unique())
    combat_labels = [c for c in COMBAT_ORDER if c in set(df_ablation["combat_label"])]

    fig, axes = plt.subplots(
        len(tasks), len(combat_labels),
        figsize=(4.5 * len(combat_labels), 0.35 * TOP_N * len(tasks) + 2),
        squeeze=False,
    )
    for i, task_id in enumerate(tasks):
        for j, combat in enumerate(combat_labels):
            ax = axes[i, j]
            sub = freq[(freq["task"].astype(str) == task_id) & (freq["combat_label"] == combat)]
            if sub.empty:
                ax.axis("off")
                continue

            pivot = sub.pivot_table(
                index="feature_short",
                columns="modality",
                values="freq",
                aggfunc="max",
                observed=False,
            )
            pivot = pivot.reindex(columns=MOD_ORDER)
            pivot = pivot.loc[pivot.max(axis=1).sort_values(ascending=False).index]

            sns.heatmap(
                pivot,
                annot=True,
                fmt=".2f",
                cmap="Blues",
                vmin=0,
                vmax=1,
                ax=ax,
                cbar=(i == 0 and j == len(combat_labels) - 1),
                cbar_kws={"label": "freq"} if (i == 0 and j == len(combat_labels) - 1) else None,
            )
            ax.set_title(f"{task_id}\n{combat}", fontsize=9)
            ax.set_xlabel("")
            ax.set_ylabel("atributo" if j == 0 else "")

    fig.suptitle("Frequência de seleção de atributos (top por task)", fontsize=12)
    plt.tight_layout()
    out = RESULTS_DIR / "ablation_features_heatmap_freq.png"
    plt.show()
    print(f"Figura: {out}")


def plot_task_summary_top_features(df_ablation: pd.DataFrame, top_n: int = 161) -> None:
    """
  Fig 3 — visão geral: por task, top atributos agregando modalidades/modelos/ComBat.
  """
    counter_by_task: dict[str, Counter] = {}
    n_by_task: dict[str, int] = {}

    for task_id, grp in df_ablation.groupby("task"):
        c: Counter = Counter()
        for feats_json in grp["selected_features"]:
            c.update(json.loads(feats_json))
        counter_by_task[str(task_id)] = c
        n_by_task[str(task_id)] = len(grp)

    tasks = sorted(counter_by_task.keys())
    fig, axes = plt.subplots(1, len(tasks), figsize=(4 * len(tasks), 6), squeeze=False)

    for ax, task_id in zip(axes.flat, tasks):
        c = counter_by_task[task_id]
        n = n_by_task[task_id]
        items = c.most_common(top_n)
        if not items:
            ax.axis("off")
            continue
        labels = [_short_feature(f) for f, _ in items]
        freqs = [cnt / n for _, cnt in items]
        ax.barh(range(len(labels)), freqs, color="#4477AA")
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=7)
        ax.set_xlim(0, 1.05)
        ax.set_xlabel("Frequência")
        ax.set_title(task_id)
        ax.invert_yaxis()

    fig.suptitle("Principais atributos por tarefa (todas modalidades)", fontsize=11)
    plt.tight_layout()
    out = RESULTS_DIR / "ablation_features_summary_by_task.png"
    plt.show()
    print(f"Figura: {out}")


# --- execução ---
df_ablation = _load_ablation()
if "selection_mode" in df_ablation.columns:
    modes = df_ablation["selection_mode"].dropna().unique()
    print("selection_mode:", list(modes))

plot_task_summary_top_features(df_ablation)
# plot_feature_heatmap(df_ablation)
# plot_top_features_by_task_modality(df_ablation)

# CSV auxiliar (opcional)
freq_all = feature_freq_table(df_ablation, top_n=TOP_N)
freq_path = RESULTS_DIR / "ablation_feature_frequency.csv"
# freq_all.to_csv(freq_path, index=False)
print(f"Tabela: {freq_path} ({len(freq_all)} linhas)")

# mrmr - minima redundancia maxima relevancia

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

RESULTS_DIR = Path("csvs/longitudinal_4_groups/ablation_results")
RESULTS_PATH = RESULTS_DIR / "ablation_results_all.csv"
SUMMARY_PATH = RESULTS_DIR / "ablation_summary.csv"

# Gráficos do resumo (legível vs tabela completa)
import seaborn as sns
from matplotlib.patches import Patch

MODEL_ORDER = ["svm", "rf", "mlp"]
MOD_ORDER = ["vol", "texture", "disp", "all"]
COMBAT_ORDER = ["sem ComBat", "ComBat"]
COLOR_MAP = {"svm": "#4477AA", "rf": "#EE6677", "mlp": "#228833"}


def _coerce_with_combat(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().isin(("true", "1", "yes"))


def _load_summary_if_needed() -> pd.DataFrame:
    if "summary" in globals() and globals().get("summary") is not None:
        return globals()["summary"]
    if "SUMMARY_PATH" not in globals():
        raise NameError("Rode a célula de config (imports + RESULTS_DIR).")
    summary_path = globals()["SUMMARY_PATH"]
    if not Path(summary_path).is_file():
        raise FileNotFoundError(f"summary ausente: {summary_path}")
    return pd.read_csv(summary_path)


summary = _load_summary_if_needed()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plot_df = summary.copy()
if "with_combat" not in plot_df.columns:
    raise KeyError("summary sem coluna with_combat")
plot_df["with_combat"] = _coerce_with_combat(plot_df["with_combat"])
plot_df["combat_label"] = plot_df["with_combat"].map({True: "ComBat", False: "sem ComBat"})
if plot_df["combat_label"].isna().any():
    raise ValueError("with_combat inválido — valores não mapeiam para ComBat/sem ComBat")

plot_df["model_key"] = pd.Categorical(
    plot_df["model_key"].astype(str), categories=MODEL_ORDER, ordered=True
)
plot_df["modality"] = pd.Categorical(
    plot_df["modality"].astype(str), categories=MOD_ORDER, ordered=True
)

tasks = sorted(plot_df["task"].astype(str).unique())
combat_labels = [c for c in COMBAT_ORDER if c in set(plot_df["combat_label"])]
if not tasks or not combat_labels:
    raise ValueError(f"Nada para plotar: tasks={tasks}, combat={combat_labels}")

# 1) Heatmap AUC: modalidade × modelo (painel por task × ComBat)
fig, axes = plt.subplots(
    len(tasks), len(combat_labels),
    figsize=(4 * len(combat_labels), 3 * len(tasks)),
    squeeze=False,
)
for i, task_id in enumerate(tasks):
    for j, combat in enumerate(combat_labels):
        ax = axes[i, j]
        sub = plot_df[(plot_df["task"].astype(str) == task_id) & (plot_df["combat_label"] == combat)]
        if sub.empty:
            ax.axis("off")
            continue
        pivot = sub.pivot_table(
            index="model_key", columns="modality", values="auc_mean", aggfunc="first", observed=False
        )
        pivot = pivot.reindex(index=MODEL_ORDER, columns=MOD_ORDER)
        sns.heatmap(
            pivot, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0.5, vmax=1.0,
            ax=ax, cbar=(i == 0 and j == len(combat_labels) - 1),
            cbar_kws={"label": "AUC"} if (i == 0 and j == len(combat_labels) - 1) else None,
        )
        ax.set_title(f"{task_id}\n{combat}", fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("modelo" if j == 0 else "")
fig.suptitle("AUC médio (5 folds)", fontsize=12)
plt.tight_layout()
out_heat = RESULTS_DIR / "ablation_summary_heatmap_auc.png"
# fig.savefig(out_heat, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura: {out_heat}")

# 2) AUC com barras + std (um gráfico por task)
for task_id in tasks:
    sub = plot_df[plot_df["task"].astype(str) == task_id]
    fig, axes = plt.subplots(1, len(combat_labels), figsize=(5 * len(combat_labels), 4), squeeze=False)
    for ax, combat in zip(axes.flat, combat_labels):
        part = sub[sub["combat_label"] == combat]
        labels, means, stds, colors = [], [], [], []
        for mod in MOD_ORDER:
            for model in MODEL_ORDER:
                row = part[(part["modality"].astype(str) == mod) & (part["model_key"].astype(str) == model)]
                if row.empty:
                    continue
                r = row.iloc[0]
                labels.append(f"{mod}\n{model}")
                means.append(float(r["auc_mean"]))
                stds.append(float(r["auc_std"]))
                colors.append(COLOR_MAP[model])
        if labels:
            ax.bar(labels, means, yerr=stds, capsize=3, color=colors, ecolor="gray")
        else:
            ax.text(0.5, 0.5, "sem dados", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(combat)
        ax.set_ylabel("AUC")
        ax.tick_params(axis="x", labelsize=7)
        ax.axhline(0.5, color="gray", ls="--", lw=0.8)
        ax.set_ylim(0.4, 1.0)
    fig.suptitle(f"Task {task_id} — AUC ± std", fontsize=11)
    plt.tight_layout()
    out = RESULTS_DIR / f"ablation_summary_bars_{task_id}.png"
    # fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figura: {out}")

# 3) Melhor config por task × modalidade
best = (
    plot_df.sort_values("auc_mean", ascending=False)
    .groupby(["task", "modality", "combat_label"], as_index=False, observed=True)
    .first()
)
fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * len(best))))
y_labels = (
    best["task"].astype(str) + " | " + best["modality"].astype(str) + " | " + best["combat_label"]
)
ax.barh(
    range(len(best)),
    best["auc_mean"],
    xerr=best["auc_std"],
    color=best["model_key"].astype(str).map(COLOR_MAP),
    capsize=3,
    ecolor="gray",
)
ax.set_yticks(range(len(best)))
ax.set_yticklabels(y_labels, fontsize=8)
ax.set_xlabel("AUC médio ± std")
ax.axvline(0.5, color="gray", ls="--", lw=0.8)
ax.set_title("Melhor modelo (maior AUC) por task × modalidade")
ax.legend(handles=[Patch(facecolor=c, label=k.upper()) for k, c in COLOR_MAP.items()])
plt.tight_layout()
out_best = RESULTS_DIR / "ablation_summary_best_auc.png"
# fig.savefig(out_best, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura: {out_best}")


In [ ]:
# Principais atributos por tarefa de classificação — complemento aos gráficos de AUC
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# alinhar com célula de gráficos AUC
RESULTS_DIR = Path("csvs/longitudinal_4_groups/ablation_results")
RESULTS_PATH = RESULTS_DIR / "ablation_results_all.csv"

MOD_ORDER = ["vol", "texture", "disp", "all"]
COMBAT_ORDER = ["sem ComBat", "ComBat"]
TOP_N = 12          # atributos por painel
FEAT_RE = re.compile(r"^hippocampus_([LR])_(T[123])_(.+)$")


def _coerce_with_combat(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().isin(("true", "1", "yes"))


def _short_feature(name: str) -> str:
    m = FEAT_RE.match(name)
    if m:
        return f"{m.group(1)} {m.group(2)} | {m.group(3)}"
    return name


def _load_ablation() -> pd.DataFrame:
    if "df_ablation" in globals() and globals()["df_ablation"] is not None:
        df = globals()["df_ablation"].copy()
    else:
        if not RESULTS_PATH.is_file():
            raise FileNotFoundError(f"CSV ausente: {RESULTS_PATH}")
        df = pd.read_csv(RESULTS_PATH)
    if "selected_features" not in df.columns:
        raise KeyError("Coluna selected_features ausente — rode ablação com runner atual.")
    df["with_combat"] = _coerce_with_combat(df["with_combat"])
    df["combat_label"] = df["with_combat"].map({True: "ComBat", False: "sem ComBat"})
    return df


def feature_freq_table(df: pd.DataFrame, top_n: int = TOP_N) -> pd.DataFrame:
    """Frequência de cada atributo em selected_features (por linha do CSV)."""
    rows = []
    for _, r in df.iterrows():
        for feat in json.loads(r["selected_features"]):
            rows.append({
                "task": r["task"],
                "modality": r["modality"],
                "combat_label": r["combat_label"],
                "feature": feat,
                "feature_short": _short_feature(feat),
            })
    if not rows:
        return pd.DataFrame(columns=["task", "modality", "combat_label", "feature", "feature_short", "count", "freq"])

    long = pd.DataFrame(rows)
    n_runs = len(df)
    agg = (
        long.groupby(["task", "modality", "combat_label", "feature", "feature_short"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
    )
    agg["freq"] = agg["count"] / n_runs
    agg = agg.sort_values(["task", "modality", "combat_label", "freq"], ascending=[True, True, True, False])
    agg["rank"] = agg.groupby(["task", "modality", "combat_label"]).cumcount() + 1
    return agg[agg["rank"] <= top_n]


def plot_top_features_by_task_modality(df_ablation: pd.DataFrame) -> None:
    """
  Fig 1 — espelha barras de AUC: 1 figura por task, painéis ComBat,
  barras horizontais = top atributos por modalidade.
  """
    freq = feature_freq_table(df_ablation, top_n=TOP_N)
    tasks = sorted(df_ablation["task"].astype(str).unique())
    combat_labels = [c for c in COMBAT_ORDER if c in set(df_ablation["combat_label"])]

    for task_id in tasks:
        sub = freq[freq["task"].astype(str) == task_id]
        if sub.empty:
            continue

        fig, axes = plt.subplots(
            1, len(combat_labels),
            figsize=(6 * len(combat_labels), 5),
            squeeze=False,
        )
        for ax, combat in zip(axes.flat, combat_labels):
            part = sub[sub["combat_label"] == combat]
            if part.empty:
                ax.axis("off")
                continue

            y_pos = 0
            yticks, ylabels = [], []
            for mod in MOD_ORDER:
                block = part[part["modality"].astype(str) == mod].sort_values("freq", ascending=False)
                if block.empty:
                    continue
                for _, row in block.iterrows():
                    ax.barh(y_pos, row["freq"], height=0.8, alpha=0.85, label=mod if y_pos == 0 else None)
                    yticks.append(y_pos)
                    ylabels.append(f"{mod} | {row['feature_short']}")
                    y_pos += 1
                y_pos += 0.4  # espaço entre modalidades

            ax.set_yticks(yticks)
            ax.set_yticklabels(ylabels, fontsize=7)
            ax.set_xlim(0, 1.05)
            ax.set_xlabel("Frequência de seleção")
            ax.set_title(combat)
            ax.axvline(0.5, color="gray", ls="--", lw=0.6, alpha=0.5)

        fig.suptitle(f"Top atributos — task {task_id}", fontsize=11)
        plt.tight_layout()
        out = RESULTS_DIR / f"ablation_features_bars_{task_id}.png"
        plt.show()
        print(f"Figura: {out}")


def plot_feature_heatmap(df_ablation: pd.DataFrame) -> None:
    """
  Fig 2 — espelha heatmap de AUC: painéis task × ComBat,
  colunas = modalidade, linhas = atributo (nome curto), cor = frequência.
  """
    freq = feature_freq_table(df_ablation, top_n=TOP_N)
    tasks = sorted(df_ablation["task"].astype(str).unique())
    combat_labels = [c for c in COMBAT_ORDER if c in set(df_ablation["combat_label"])]

    fig, axes = plt.subplots(
        len(tasks), len(combat_labels),
        figsize=(4.5 * len(combat_labels), 0.35 * TOP_N * len(tasks) + 2),
        squeeze=False,
    )
    for i, task_id in enumerate(tasks):
        for j, combat in enumerate(combat_labels):
            ax = axes[i, j]
            sub = freq[(freq["task"].astype(str) == task_id) & (freq["combat_label"] == combat)]
            if sub.empty:
                ax.axis("off")
                continue

            pivot = sub.pivot_table(
                index="feature_short",
                columns="modality",
                values="freq",
                aggfunc="max",
                observed=False,
            )
            pivot = pivot.reindex(columns=MOD_ORDER)
            pivot = pivot.loc[pivot.max(axis=1).sort_values(ascending=False).index]

            sns.heatmap(
                pivot,
                annot=True,
                fmt=".2f",
                cmap="Blues",
                vmin=0,
                vmax=1,
                ax=ax,
                cbar=(i == 0 and j == len(combat_labels) - 1),
                cbar_kws={"label": "freq"} if (i == 0 and j == len(combat_labels) - 1) else None,
            )
            ax.set_title(f"{task_id}\n{combat}", fontsize=9)
            ax.set_xlabel("")
            ax.set_ylabel("atributo" if j == 0 else "")

    fig.suptitle("Frequência de seleção de atributos (top por task)", fontsize=12)
    plt.tight_layout()
    out = RESULTS_DIR / "ablation_features_heatmap_freq.png"
    plt.show()
    print(f"Figura: {out}")


def plot_task_summary_top_features(df_ablation: pd.DataFrame, top_n: int = 10) -> None:
    """
  Fig 3 — visão geral: por task, top atributos agregando modalidades/modelos/ComBat.
  """
    counter_by_task: dict[str, Counter] = {}
    n_by_task: dict[str, int] = {}

    for task_id, grp in df_ablation.groupby("task"):
        c: Counter = Counter()
        for feats_json in grp["selected_features"]:
            c.update(json.loads(feats_json))
        counter_by_task[str(task_id)] = c
        n_by_task[str(task_id)] = len(grp)

    tasks = sorted(counter_by_task.keys())
    fig, axes = plt.subplots(1, len(tasks), figsize=(4 * len(tasks), 6), squeeze=False)

    for ax, task_id in zip(axes.flat, tasks):
        c = counter_by_task[task_id]
        n = n_by_task[task_id]
        items = c.most_common(top_n)
        if not items:
            ax.axis("off")
            continue
        labels = [_short_feature(f) for f, _ in items]
        freqs = [cnt / n for _, cnt in items]
        ax.barh(range(len(labels)), freqs, color="#4477AA")
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=7)
        ax.set_xlim(0, 1.05)
        ax.set_xlabel("Frequência")
        ax.set_title(task_id)
        ax.invert_yaxis()

    fig.suptitle("Principais atributos por tarefa (todas modalidades)", fontsize=11)
    plt.tight_layout()
    out = RESULTS_DIR / "ablation_features_summary_by_task.png"
    plt.show()
    print(f"Figura: {out}")


# --- execução ---
df_ablation = _load_ablation()
if "selection_mode" in df_ablation.columns:
    modes = df_ablation["selection_mode"].dropna().unique()
    print("selection_mode:", list(modes))

plot_task_summary_top_features(df_ablation)
plot_feature_heatmap(df_ablation)
plot_top_features_by_task_modality(df_ablation)

# CSV auxiliar (opcional)
freq_all = feature_freq_table(df_ablation, top_n=TOP_N)
freq_path = RESULTS_DIR / "ablation_feature_frequency.csv"
freq_all.to_csv(freq_path, index=False)
print(f"Tabela: {freq_path} ({len(freq_all)} linhas)")